In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report

from data_loading_code import load_data

In [16]:
class MLPClassifier(nn.Module):
    """Simple feed-forward neural network for binary text classification."""

    def __init__(self, input_dim: int, hidden_dims: list, num_classes: int = 2, dropout: float = 0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [17]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct = 0.0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0.0, 0
    all_preds, all_labels = [], []
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        total_loss += criterion(logits, y).item() * len(y)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(y.cpu().tolist())
    n = len(loader.dataset)
    return total_loss / n, correct / n, all_preds, all_labels

In [18]:
HIDDEN_DIMS  = [256, 128]
DROPOUT      = 0.3
BATCH_SIZE   = 64
EPOCHS       = 20
LR           = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

X_train, y_train, X_val, y_val, vocab_size = load_data()
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Vocab: {vocab_size}")

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=BATCH_SIZE)

Using device: cpu
Train: torch.Size([900, 7277]) | Val: torch.Size([100, 7277]) | Vocab: 7277


In [19]:
input_dim = X_train.shape[1]
model     = MLPClassifier(input_dim, HIDDEN_DIMS, dropout=DROPOUT).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print(model)

MLPClassifier(
  (net): Sequential(
    (0): Linear(in_features=7277, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=2, bias=True)
  )
)


In [20]:
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch:02d}/{EPOCHS}  "
          f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
          f"val_loss={vl_loss:.4f}  val_acc={vl_acc:.4f}")

Epoch 01/20  train_loss=0.6928  train_acc=0.5033  val_loss=0.6845  val_acc=0.5200
Epoch 02/20  train_loss=0.6528  train_acc=0.9056  val_loss=0.6186  val_acc=0.8300
Epoch 03/20  train_loss=0.4800  train_acc=0.9878  val_loss=0.4584  val_acc=0.8000
Epoch 04/20  train_loss=0.1940  train_acc=0.9967  val_loss=0.3370  val_acc=0.8300
Epoch 05/20  train_loss=0.0349  train_acc=0.9989  val_loss=0.3272  val_acc=0.8300
Epoch 06/20  train_loss=0.0058  train_acc=1.0000  val_loss=0.3596  val_acc=0.8300
Epoch 07/20  train_loss=0.0023  train_acc=1.0000  val_loss=0.3875  val_acc=0.8100
Epoch 08/20  train_loss=0.0012  train_acc=1.0000  val_loss=0.4032  val_acc=0.8200
Epoch 09/20  train_loss=0.0008  train_acc=1.0000  val_loss=0.4136  val_acc=0.8200
Epoch 10/20  train_loss=0.0006  train_acc=1.0000  val_loss=0.4289  val_acc=0.8100
Epoch 11/20  train_loss=0.0004  train_acc=1.0000  val_loss=0.4436  val_acc=0.8100
Epoch 12/20  train_loss=0.0003  train_acc=1.0000  val_loss=0.4546  val_acc=0.8100
Epoch 13/20  tra

In [21]:
_, _, preds, labels = evaluate(model, val_loader, criterion, device)
print("\nClassification Report (Validation):")
print(classification_report(labels, preds, target_names=["Negative", "Positive"]))


Classification Report (Validation):
              precision    recall  f1-score   support

    Negative       0.75      0.89      0.82        47
    Positive       0.89      0.74      0.80        53

    accuracy                           0.81       100
   macro avg       0.82      0.81      0.81       100
weighted avg       0.82      0.81      0.81       100

